# Constructing and Testing Scraper Pipelines

Use this notebook to _create_, _test_, and _validate_ each particular scraping function (for each data source). The final functions will be added as methods to the _EventScraper_ class.

In [1]:
# import required libraries
import requests
import pandas as pd
from bs4 import BeautifulSoup
from scrapy import Selector
import pyperclip
import os
import re

## Tacoma Dome

---

In [2]:
# read in the `exploration` html directly from a URL and prettify it using BeautifulSoup, then write it to a file
link = "https://www.tacomadome.org/events"

# set the tacoma dome location as the default location for the events
location = "2727 E D St, Tacoma, WA 98421"
loc_link = "https://maps.app.goo.gl/bT8bE5ERX2Jdbybp8" # in the future we will use Google Maps API to get the location link, but for now we will hardcode it

# get the HTML content of the page
response = requests.get(link)
soup = BeautifulSoup(response.content, 'html.parser')

# filter to only the section of interest, which is the table containing the data we want to extract
target_table = soup.find("div", class_="full_column non-widget-area") # IMPORTANT: this is the div that contains the table we want to extract

# strip out unnecessary scripts and styles
for script in target_table(["script", "style"]):
    script.decompose()

# find all blocks where the class has the pattern r"eventItem
pat = re.compile(r"^eventItem.*clearfix$")

# find all divs with the class that matches the pattern
event_items = target_table.find_all(class_=pat)

# create a dictionary to hold the event data
event_dict = {}
td_df = pd.DataFrame()

# parse each event to get name, link to details, date (range), description, image link, location, price, and ticket link (if available)
for event in event_items:
    # get the name of the event
    try:
        event_dict["name"] = event.find("h3").get_text(strip=True)
    except AttributeError:
        event_dict["name"] = None

    # get the link to the details page
    try:
        event_dict["details_link"] = event.find("a")["href"]
    except AttributeError:
        event_dict["details_link"] = None

    # get the date range of the event
    try:
        event_dict["date_range"] = event.find("div", class_="date").get("aria-label").strip()
    except AttributeError:
        event_dict["date_range"] = None

    # get the description of the event
    try:
        event_dict["description"] = event.find("h4").get_text(strip=True)
    except AttributeError:
        event_dict["description"] = None

    # get the image link of the event
    try:
        event_dict["image_link"] = event.find("img")["src"]
    except AttributeError:
        event_dict["image_link"] = None
        
    # get the ticket link of the event (if available)
    try:
        event_dict["ticket_link"] = event.find("a", class_="tickets onsalenow").get("href").strip()
    except AttributeError:
        event_dict["ticket_link"] = None

    # concatenate the event_dict to the dataframe
    td_df = pd.concat([td_df, pd.DataFrame([event_dict])], ignore_index=True)

# add the location and location link to the dataframe (invariant for all events at the Tacoma Dome)
td_df["location"] = location
td_df["location_link"] = loc_link

## Emerald Queen

---

In [3]:
# read in the `exploration` html directly from a URL and prettify it using BeautifulSoup, then write it to a file
link = "https://emeraldqueen.com/tickets/"

# set the tacoma dome location as the default location for the events
location = "Pacific Hwy E, Fife, WA 98424"
loc_link = "https://maps.app.goo.gl/8SuRqvAfA7htoRZN6" # in the future we will use Google Maps API to get the location link, but for now we will hardcode it

# get the HTML content of the page
response = requests.get(link)
soup = BeautifulSoup(response.content, 'html.parser')

# filter to only the section of interest, which is the table containing the data we want to extract
target_table = soup.find("section", class_="content page-925 moto-section") # IMPORTANT: this is the div that contains the table we want to extract

# find all divs with the class that matches the pattern
event_items = target_table.find_all("div", class_="moto-widget moto-widget-row moto-spacing-top-medium moto-spacing-right-auto moto-spacing-bottom-medium moto-spacing-left-auto")

# create a dictionary to hold the event data
event_dict = {}
eqc_df = pd.DataFrame()

In [4]:
for event in event_items:
    print(event.prettify())

<div class="moto-widget moto-widget-row moto-spacing-top-medium moto-spacing-right-auto moto-spacing-bottom-medium moto-spacing-left-auto" data-bg-position="left top" data-grid-type="sm" data-spacing="mama" data-visible-on="-" data-widget="row" style="">
 <div class="container-fluid">
  <div class="row" data-container="container">
   <div class="moto-widget moto-widget-row__column moto-cell col-sm-6 moto-spacing-top-auto moto-spacing-right-auto moto-spacing-bottom-auto moto-spacing-left-auto" data-bg-position="left top" data-container="container" data-enabled-side-spacing="false" data-spacing="aaaa" data-widget="row.column" style="">
    <div class="moto-widget moto-widget-row moto-spacing-top-auto moto-spacing-right-auto moto-spacing-bottom-auto moto-spacing-left-auto" data-bg-position="left top" data-grid-type="sm" data-spacing="aaaa" data-visible-on="-" data-widget="row" style="">
     <div class="container-fluid">
      <div class="row" data-container="container">
       <div class

In [5]:
# parse each event to get name, link to details, date (range), description, image link, location, price, and ticket link (if available)
for event in event_items:
    # get the name of the event
    try:
        event_dict["name"] = event.find("h3").get_text(strip=True)
    except AttributeError:
        event_dict["name"] = None

    # get the link to the details page
    try:
        event_dict["details_link"] = event.find("a")["href"]
    except AttributeError:
        event_dict["details_link"] = None

    # get the date range of the event
    try:
        event_dict["date_range"] = event.find("div", class_="date").get("aria-label").strip()
    except AttributeError:
        event_dict["date_range"] = None

    # get the description of the event
    try:
        event_dict["description"] = event.find("h4").get_text(strip=True)
    except AttributeError:
        event_dict["description"] = None

    # get the image link of the event
    try:
        event_dict["image_link"] = event.find("img")["src"]
    except AttributeError:
        event_dict["image_link"] = None
        
    # get the ticket link of the event (if available)
    try:
        event_dict["ticket_link"] = event.find("a", class_="tickets onsalenow").get("href").strip()
    except AttributeError:
        event_dict["ticket_link"] = None

    # concatenate the event_dict to the dataframe
    eqc_df = pd.concat([eqc_df, pd.DataFrame([event_dict])], ignore_index=True)

# add the location and location link to the dataframe (invariant for all events at the Tacoma Dome)
eqc_df["location"] = location
eqc_df["location_link"] = loc_link